# 2. Provision a Microsoft Entra agent identity

Create the identity objects for the `LearningAssistant` from notebook 01. This notebook uses Python, interactive Microsoft Graph sign-in, and explicit REST requests so you can see what is being provisioned.

| Object | What it represents |
| --- | --- |
| Agent identity blueprint | An agent type. Runtime credentials will live here, not on individual identities. |
| Blueprint principal | The blueprint's service principal in this tenant. Graph creation of a blueprint does not automatically create it. |
| Agent identity | A particular agent instance with its own directory identity and permission grants. |

**This creates real Entra directory objects.** It does not configure runtime credentials, grant the agent access to Graph or Foundry, publish a Foundry agent, create an agent user/mailbox, or register the agent in Agent 365. Notebook 01 will still call Foundry as you.

## 0. Prepare a dedicated provisioning client

In your tenant, create or reuse a normal **App registration** for this notebook's interactive sign-in. This is a provisioning tool, **not** the agent blueprint and **not** the agent identity.

**Script option:** run the included bootstrap script from PowerShell 7. It creates/reuses the app, configures the redirect and delegated permissions below, and creates/reuses its service principal. No secret is created.

```powershell
Install-Module Microsoft.Graph.Authentication -Scope CurrentUser
.\New-AgentIdentityProvisioningClient.ps1 -TenantId YOUR-TENANT-GUID -WhatIf
.\New-AgentIdentityProvisioningClient.ps1 -TenantId YOUR-TENANT-GUID
```

The script signs in through Microsoft Graph PowerShell with `Application.ReadWrite.All`; your account must be authorized to manage app registrations. It prints the two `.env` values to copy, without changing the file. The preview authenticates and reads Graph but makes no directory changes.

Admin consent is separate and is **not granted by default**. An authorized administrator (normally Cloud Application Administrator or Application Administrator) can grant it in Entra or explicitly run:

```powershell
.\New-AgentIdentityProvisioningClient.ps1 -TenantId YOUR-TENANT-GUID -GrantAdminConsent
```

That switch additionally requests `DelegatedPermissionGrant.ReadWrite.All` for the bootstrap session and grants tenant-wide consent for this notebook's seven delegated permissions. Reruns preserve existing redirects, API permissions, and consent scopes. If multiple apps share the name, use `-ApplicationObjectId` to choose one. After a replication-related failure, wait and rerun with the object ID printed by the script.

**Portal option:** perform the following equivalent setup manually:

1. Use a single-tenant app registration, for example `AgentIdentityNotebookProvisioner`.
2. Under **Authentication**, add the **Mobile and desktop applications** platform with redirect URI `http://localhost:8400`. This notebook uses interactive authorization-code flow with PKCE; no client secret is required. Do not configure this redirect as a Web/confidential client.
3. Under **API permissions**, add the following Microsoft Graph **delegated** permissions. Have an authorized administrator grant consent for the required permissions.

| Delegated permission | Why this notebook needs it |
| --- | --- |
| `User.Read` | Identify the signed-in user, used as sponsor and owner. |
| `AgentIdentityBlueprint.Create` | Create the blueprint. |
| `AgentIdentityBlueprint.Read.All` | Read and verify/reuse the blueprint. |
| `AgentIdentityBlueprintPrincipal.Create` | Create the blueprint principal. |
| `AgentIdentityBlueprintPrincipal.Read.All` | Read and verify/reuse the principal. |
| `AgentIdentity.Create.All` | Create an agent identity under the blueprint. |
| `AgentIdentity.Read.All` | Read and verify/reuse the agent identity. |

**Client permissions and your user permissions are different.** Your tenant must allow you to create blueprints, or you need an appropriate Agent ID role. A blueprint creator is automatically an owner; owners can create identities under their blueprint without an additional Agent ID role. Reusing objects you do not own can require Agent ID Administrator. Activate any required PIM role before signing in.

Use a work/school account in a tenant with Entra Agent ID available. Do not substitute `AzureCliCredential` or `DefaultAzureCredential`: the Azure CLI Graph permission set is not appropriate for these Agent ID provisioning APIs.

### Visual guide: why the permissions list can be empty

The screenshot below shows the **provisioning client's enterprise application**, not its app registration and not the agent blueprint. These are different views of the setup.

![Provisioning client's Enterprise Application Permissions page before admin consent, showing an empty Admin consent tab and the Application registration link](docs/images/provisioning-client-permissions.png)

**What you are seeing:** the **Admin consent** tab lists permissions already granted for this application in the tenant. An empty list does not mean the script failed to configure the requested permissions.

**What to do:**

1. Click the blue **Application registration** link shown above the tabs, then open **API permissions**. Verify that its client ID matches `ENTRA_PROVISIONING_CLIENT_ID` in `.env`.
2. Check that the seven delegated Microsoft Graph permissions listed above are configured. This is where requested permissions are shown; the bootstrap script adds them but does not grant consent by default.
3. Have an authorized administrator select **Grant admin consent**, or run the bootstrap script with `-GrantAdminConsent`. After consent succeeds, restart the notebook kernel and sign in again.

**Why we do this:** configuring permissions declares what the provisioning tool needs; granting consent authorizes it to request those permissions. Your signed-in user's Agent ID role/ownership is still checked separately. Neither step creates the blueprint: the provisioning cells below do that.

## 1. Configure the notebook

Use this project's `.venv` kernel. In PowerShell, install dependencies if needed:

```powershell
uv pip install --python .venv\Scripts\python.exe -r requirements.txt
```

Restart the kernel after dependency updates. Fill in these values in the existing `.env`, preserving notebook 01's Foundry settings:

```dotenv
ENTRA_TENANT_ID=YOUR-TENANT-GUID
ENTRA_PROVISIONING_CLIENT_ID=YOUR-PROVISIONING-APP-CLIENT-GUID
AGENT_BLUEPRINT_NAME=LearningAssistantBlueprint
AGENT_IDENTITY_NAME=LearningAssistantLocal
```

If you already have a blueprint or identity, optionally set `ENTRA_BLUEPRINT_OBJECT_ID` and `ENTRA_AGENT_IDENTITY_OBJECT_ID` to their **directory object IDs**, not application/client IDs. The notebook verifies their types and relationships before reuse. For an existing agent identity, provide its existing blueprint too.

Run with the project directory as the working directory. `.env` is ignored by Git. Existing process environment variables take precedence; restart the kernel after changing `.env`.

In [14]:
import os
from urllib.parse import urlencode
from uuid import UUID

import requests
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv

In [15]:
load_dotenv(".env")

tenant_id = os.getenv("ENTRA_TENANT_ID", "").strip()
provisioning_client_id = os.getenv("ENTRA_PROVISIONING_CLIENT_ID", "").strip()
blueprint_name = os.getenv("AGENT_BLUEPRINT_NAME", "LearningAssistantBlueprint").strip()
agent_name = os.getenv("AGENT_IDENTITY_NAME", "LearningAssistantLocal").strip()
blueprint_object_id = os.getenv("ENTRA_BLUEPRINT_OBJECT_ID", "").strip()
agent_identity_id = os.getenv("ENTRA_AGENT_IDENTITY_OBJECT_ID", "").strip()

tenant_id = str(UUID(tenant_id))
provisioning_client_id = str(UUID(provisioning_client_id))
if not blueprint_name or not agent_name:
    raise ValueError("Set nonempty AGENT_BLUEPRINT_NAME and AGENT_IDENTITY_NAME in .env.")
if blueprint_object_id:
    blueprint_object_id = str(UUID(blueprint_object_id))
if agent_identity_id:
    agent_identity_id = str(UUID(agent_identity_id))

print("Tenant:", tenant_id)
print("Blueprint:", blueprint_name)
print("Agent instance:", agent_name)

Tenant: 7b3b2559-0c78-44e7-8c57-8455603aca36
Blueprint: LearningAssistantBlueprint
Agent instance: LearningAssistantLocal


## 2. Choose which identities to reuse

Keep the IDs in ordinary Python variables. Reuse an explicit blueprint ID from `.env`, or look up the blueprint by `AGENT_BLUEPRINT_NAME`. Create it only when no match exists. No checkpoint file is read or written.

Rerunning a provisioning cell in the same kernel reuses the ID it already assigned. A fresh kernel can find the blueprint by name, but reusing the child agent identity still requires `ENTRA_AGENT_IDENTITY_OBJECT_ID` and its `ENTRA_BLUEPRINT_OBJECT_ID` in `.env`.

**Blueprint names are not unique.** Multiple matches stop the notebook and require an explicit blueprint ID. The lookup/create sequence is not atomic: concurrent runs or replication delays can still cause duplicates. Do not run provisioning concurrently. Use a fresh kernel when switching tenants.

In [16]:
if agent_identity_id and not blueprint_object_id:
    raise ValueError("Provide ENTRA_BLUEPRINT_OBJECT_ID when reusing an existing agent identity.")

print("Blueprint:", blueprint_object_id or f"look up by name: {blueprint_name}")
print("Existing agent identity:", agent_identity_id or "will create")

Blueprint: look up by name: LearningAssistantBlueprint
Existing agent identity: will create


## 3. Sign in as the provisioning user

The next cell opens a browser the first time Graph needs a token. Sign in with the account authorized above. This is **your provisioning identity**, not the agent's runtime identity.

The small helper gets a fresh/cached token for each request, limits calls to Microsoft Graph v1.0, and reports Graph errors without printing tokens. POST requests are not automatically retried.

In [17]:
graph_base = "https://graph.microsoft.com/v1.0"
graph_scopes = [f"https://graph.microsoft.com/{permission}" for permission in (
    "User.Read",
    "AgentIdentityBlueprint.Create",
    "AgentIdentityBlueprint.Read.All",
    "AgentIdentityBlueprintPrincipal.Create",
    "AgentIdentityBlueprintPrincipal.Read.All",
    "AgentIdentity.Create.All",
    "AgentIdentity.Read.All",
)]

graph_credential = InteractiveBrowserCredential(
    tenant_id=tenant_id,
    client_id=provisioning_client_id,
    redirect_uri="http://localhost:8400",
)
http = requests.Session()

def graph_request(method, path, body=None, *, allow_not_found=False):
    if not path.startswith("/") or path.startswith("//"):
        raise ValueError("Use a relative Microsoft Graph path beginning with a single /.")
    if method != "GET" and not provisioning_confirmed:
        raise RuntimeError("Run the confirmation cell before making Graph changes.")
    access_token = graph_credential.get_token(*graph_scopes)
    response = http.request(
        method,
        graph_base + path,
        headers={
            "Authorization": f"Bearer {access_token.token}",
            "Content-Type": "application/json",
            "OData-Version": "4.0",
        },
        json=body,
        timeout=30,
        allow_redirects=False,
    )
    if response.status_code == 404 and allow_not_found:
        return None
    if not 200 <= response.status_code < 300:
        try:
            error = response.json().get("error", {})
            detail = f"{error.get('code', '')}: {error.get('message', '')}"
        except ValueError:
            detail = "Non-JSON error response."
        raise RuntimeError(f"Graph {method} {path} failed ({response.status_code}): {detail}; request-id={response.headers.get('request-id', 'unknown')}")
    return response.json() if response.content else None

provisioning_confirmed = False
me = graph_request("GET", "/me?$select=id,displayName")
sponsor_id = str(UUID(me["id"]))
sponsor_reference = f"{graph_base}/users/{sponsor_id}"
print("Signed-in sponsor/owner:", me["displayName"])
print("Sponsor object ID:", sponsor_id)

Signed-in sponsor/owner: Megan Demo
Sponsor object ID: 03283c82-156c-4564-9a0a-0f00af835b4d


## 4. Review and explicitly allow provisioning

The signed-in user will be sponsor and owner of a **new** blueprint and sponsor of a **new** agent identity. Reused objects keep their existing sponsors/owners. This notebook will create only missing objects and will not add credentials, permissions, or endpoints.

Type `PROVISION` to proceed. Any other answer stops the workflow. Do not use an unattended Run All if you have not reviewed the tenant and names.

In [18]:
print(f"Tenant: {tenant_id}\nBlueprint: {blueprint_name}\nAgent identity: {agent_name}")
print("Blueprint:", blueprint_object_id or f"look up by name: {blueprint_name}")
print("Existing agent identity:", agent_identity_id or "will create")
provisioning_confirmed = input("Type PROVISION to allow creation of missing identity objects: ").strip() == "PROVISION"
if not provisioning_confirmed:
    raise RuntimeError("Provisioning cancelled. No directory changes have been made.")

Tenant: 7b3b2559-0c78-44e7-8c57-8455603aca36
Blueprint: LearningAssistantBlueprint
Agent identity: LearningAssistantLocal
Blueprint: look up by name: LearningAssistantBlueprint
Existing agent identity: will create


## 5. Create or reuse the blueprint

The blueprint is an application object. Its **`id`** is the directory object ID used to read/manage the application. Its **`appId`** is the application/client ID used when creating its principal and agent identities. These IDs are not interchangeable.

We use the typed Agent ID endpoint, not an ordinary application registration. When no object ID is supplied, filter blueprints by display name: reuse one match, create on zero matches, and stop on multiple matches. An explicit object ID takes precedence over the name.

In [19]:
if not blueprint_object_id:
    escaped_name = blueprint_name.replace("'", "''")
    query = urlencode({"$filter": f"displayName eq '{escaped_name}'", "$select": "id,appId,displayName", "$top": 2})
    lookup_path = f"/applications/microsoft.graph.agentIdentityBlueprint?{query}"
    matches = []
    while lookup_path and len(matches) < 2:
        page = graph_request("GET", lookup_path)
        matches.extend(page["value"])
        next_link = page.get("@odata.nextLink")
        if next_link and not next_link.startswith(graph_base + "/"):
            raise ValueError("Unexpected Graph pagination URL.")
        lookup_path = next_link[len(graph_base):] if next_link else None
    if len(matches) > 1:
        raise ValueError("Multiple blueprints have this name. Set ENTRA_BLUEPRINT_OBJECT_ID to choose one.")
    if matches:
        blueprint_object_id = str(UUID(matches[0]["id"]))

if blueprint_object_id:
    blueprint = graph_request("GET", f"/applications/{blueprint_object_id}/microsoft.graph.agentIdentityBlueprint")
    print("Reused blueprint:", blueprint["displayName"])
else:
    blueprint = graph_request("POST", "/applications/microsoft.graph.agentIdentityBlueprint", {
        "displayName": blueprint_name,
        "sponsors@odata.bind": [sponsor_reference],
        "owners@odata.bind": [sponsor_reference],
    })
    print("Created blueprint:", blueprint["displayName"])

blueprint_object_id = str(UUID(blueprint["id"]))
blueprint_app_id = str(UUID(blueprint["appId"]))
print("Blueprint object ID:", blueprint_object_id)
print("Blueprint application/client ID:", blueprint_app_id)

Created blueprint: LearningAssistantBlueprint
Blueprint object ID: 67ea0527-3de2-4b33-9e36-247b394eb1ff
Blueprint application/client ID: 67ea0527-3de2-4b33-9e36-247b394eb1ff


## 6. Create or reuse the blueprint principal

This is mandatory before creating an agent identity. Even when reusing a blueprint, check for its principal by the blueprint's `appId` and create it only if Graph returns 404. Other errors must not be mistaken for a missing principal.

Entra replication can briefly delay visibility. If this step fails just after blueprint creation, wait and rerun this cell; do not create another blueprint.

In [20]:
principal = graph_request(
    "GET",
    f"/servicePrincipals(appId='{blueprint_app_id}')/microsoft.graph.agentIdentityBlueprintPrincipal",
    allow_not_found=True,
)
if principal is None:
    principal = graph_request("POST", "/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal", {
        "appId": blueprint_app_id,
    })
    print("Created blueprint principal.")
else:
    print("Reused existing blueprint principal.")

if principal["appId"] != blueprint_app_id:
    raise ValueError("The principal does not belong to this blueprint.")
blueprint_principal_id = str(UUID(principal["id"]))
print("Blueprint principal object ID:", blueprint_principal_id)

Created blueprint principal.
Blueprint principal object ID: 1e2b8345-e929-48c2-9c9a-bfe2614af4b6


## 7. Create or reuse the agent identity

The blueprint defines the **type of agent** (`LearningAssistantBlueprint`). This step creates the identity for **one particular agent instance** (`LearningAssistantLocal`). That identity is what you can later grant access to resources such as Blob Storage.

Entra stores this instance identity as a special kind of **service principal**: a directory record representing software rather than a human user. You do **not** create another app registration for each agent instance; the instance is linked to the blueprint we already created.

### Which blueprint ID connects the instance to its blueprint?

The blueprint has two different IDs, returned in step 5:

| Notebook variable | Graph property | Used for |
| --- | --- | --- |
| `blueprint_object_id` | `blueprint["id"]` | Reading or managing the blueprint's own directory record. |
| `blueprint_app_id` | `blueprint["appId"]` | Identifying the parent blueprint when creating this agent instance. |

Despite its name, the request field **`agentIdentityBlueprintId` expects the blueprint's Application (client) ID**. That is why the code below uses:

```python
{"agentIdentityBlueprintId": blueprint_app_id}
```

This identity represents one instance of the local learning agent. It has no runtime credentials or resource access configured by this notebook.

In [21]:
if agent_identity_id:
    agent_identity = graph_request("GET", f"/servicePrincipals/{agent_identity_id}/microsoft.graph.agentIdentity")
    print("Reused agent identity:", agent_identity["displayName"])
else:
    agent_identity = graph_request("POST", "/servicePrincipals/microsoft.graph.agentIdentity", {
        "displayName": agent_name,
        "agentIdentityBlueprintId": blueprint_app_id,
        "sponsors@odata.bind": [sponsor_reference],
    })
    print("Created agent identity:", agent_identity["displayName"])

if agent_identity["agentIdentityBlueprintId"] != blueprint_app_id:
    raise ValueError("The agent identity belongs to a different blueprint.")
agent_identity_id = str(UUID(agent_identity["id"]))
print("Agent identity object ID:", agent_identity_id)

Created agent identity: LearningAssistantLocal
Agent identity object ID: b036c639-4651-4350-9dcc-c3a5c189843c


## 8. Read back all three objects and verify the relationships

A successful creation response is not the final check. Read the objects from Graph and check that the principal and identity reference the intended blueprint. If a newly created object is temporarily unavailable, wait and rerun this read-only cell.

In [22]:
verified_blueprint = graph_request("GET", f"/applications/{blueprint_object_id}/microsoft.graph.agentIdentityBlueprint")
verified_principal = graph_request("GET", f"/servicePrincipals/{blueprint_principal_id}/microsoft.graph.agentIdentityBlueprintPrincipal")
verified_identity = graph_request("GET", f"/servicePrincipals/{agent_identity_id}/microsoft.graph.agentIdentity")

if not (
    verified_blueprint["appId"] == blueprint_app_id
    and verified_principal["appId"] == blueprint_app_id
    and verified_identity["agentIdentityBlueprintId"] == blueprint_app_id
):
    raise ValueError("Identity relationship verification failed.")

print("Verified blueprint -> blueprint principal -> agent identity.")
print("Copy these values into .env to reuse the objects and continue with notebook 03:")
print(f"ENTRA_BLUEPRINT_OBJECT_ID={blueprint_object_id}")
print(f"ENTRA_BLUEPRINT_CLIENT_ID={blueprint_app_id}")
print(f"ENTRA_AGENT_IDENTITY_OBJECT_ID={agent_identity_id}")

Verified blueprint -> blueprint principal -> agent identity.
Copy these values into .env to reuse the objects and continue with notebook 03:
ENTRA_BLUEPRINT_OBJECT_ID=67ea0527-3de2-4b33-9e36-247b394eb1ff
ENTRA_BLUEPRINT_CLIENT_ID=67ea0527-3de2-4b33-9e36-247b394eb1ff
ENTRA_AGENT_IDENTITY_OBJECT_ID=b036c639-4651-4350-9dcc-c3a5c189843c


## 9. Close local clients (keep the provisioned objects)

Closing connections does not delete anything in Entra. No deletion commands are included to avoid accidentally removing a blueprint or its identities. After closing, rerun the Graph-client cell before further Graph calls.

In [23]:
http.close()
graph_credential.close()

## What comes next?

You have provisioned identity objects, but the agent cannot yet authenticate as that identity. The next lesson can configure a blueprint credential, explain the two-step agent token exchange, and grant the agent narrowly scoped access to a chosen resource. For production Azure hosting, prefer managed identity plus workload identity federation instead of a client secret.

Agent 365 registration and observability are separate from these Entra objects. A blueprint by itself does not change notebook 01's `AzureCliCredential` or register the agent in the Agent 365 catalog.

### Moving provisioning to CI/CD

This notebook's interactive sign-in is for learning, not unattended pipelines. Use a **dedicated app-only provisioning identity** with workload identity federation (pipeline OIDC), admin-consented Graph **application** permissions, and the `https://graph.microsoft.com/.default` token scope. Supply a sponsor user object ID explicitly instead of calling `/me`. Keep the provisioning identity separate from the agent runtime identity and bootstrap its federated trust/admin consent before running the pipeline.

Move provisioning into an idempotent script or tool, persist object IDs in durable pipeline state, and serialize provisioning runs. Python variables are sufficient for this lesson, not for remembering provisioning across pipeline runs. No pipeline or federated credential is created by this notebook.

### Troubleshooting

- **UUID/configuration error:** fill in the tenant and provisioning-client GUIDs in `.env`; restart the kernel after edits.
- **AADSTS50011 / redirect mismatch:** configure the provisioning app's Mobile and desktop redirect URI as `http://localhost:8400`.
- **Consent / invalid scope error:** check the delegated Graph permissions and admin consent on the provisioning app, not on the new blueprint.
- **403:** check your user role/ownership, PIM activation, tenant Agent ID availability, and the app's delegated permissions. Do not treat this as a missing object or grant broad directory access as a workaround.
- **400 / principal missing after blueprint creation:** allow Entra replication time, then rerun the principal cell using the blueprint ID already in memory.
- **404 for an existing ID:** check the tenant and Entra object; wait if it was just created. The notebook does not recreate missing objects when an ID was provided.
- **Partial provisioning failure:** rerun the failed cell while the kernel still holds the successful IDs. Before restarting or rerunning configuration, copy the printed IDs into .env. If the response was lost, inspect Entra rather than relying on display names.

Clear notebook outputs before sharing: they can contain directory IDs and the signed-in user's display name. Tokens are never intentionally displayed or persisted.

### References

- [Create an agent identity blueprint](https://learn.microsoft.com/entra/agent-id/create-blueprint)
- [Create an agent identity](https://learn.microsoft.com/graph/api/agentidentity-post?view=graph-rest-1.0)
- [Read a blueprint](https://learn.microsoft.com/graph/api/agentidentityblueprint-get?view=graph-rest-1.0)
- [List/filter blueprints](https://learn.microsoft.com/graph/api/agentidentityblueprint-list?view=graph-rest-1.0)
- [Read a blueprint principal](https://learn.microsoft.com/graph/api/agentidentityblueprintprincipal-get?view=graph-rest-1.0)
- [Read an agent identity](https://learn.microsoft.com/graph/api/agentidentity-get?view=graph-rest-1.0)
- [Interactive Graph authentication](https://learn.microsoft.com/graph/sdks/choose-authentication-providers#interactive-provider)